In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 14 consolidation — Make camera equations reversible only when the information exists

Read `sessions/14_session.md` first. This is an executed reference, not an assessment of you.

**Evidence:** Real calibrated stereo-reference points plus a hand-checkable pinhole control.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys, json
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT/'data/raw/iris.csv').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Start Jupyter in the extracted listening_to_shape_lab folder.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))


## Step 1

Use a hand-checkable pinhole example to expose ambiguity along one ray.

Before running: write your expected result and name one way this check could fail.

In [3]:
from shape_lab.geometry import project, unproject, exp_so3, make_transform, transform_points, inverse_transform
K = np.array([[1000.,0,320],[0,1000,240],[0,0,1.]])
pair = np.array([[.2,.1,2.],[.4,.2,4.]])
uv, z = project(pair, K)
assert np.allclose(uv, [[420,290],[420,290]])
assert np.allclose(unproject(uv, z, K), pair)
print('same pixels, different depths:', uv, z)

same pixels, different depths: [[420. 290.]
 [420. 290.]] [2. 4.]


## Step 2

Run the same consistency checks on a sample from the real stereo reference.

Before running: write your expected result and name one way this check could fail.

In [4]:
from shape_lab.stereo import reference_sample, point_cloud
_, _, disparity, calibration = reference_sample()
X, rc = point_cloud(disparity, calibration, stride=60)
uv, z = project(X, calibration['K_left'])
assert np.allclose(uv, rc[:,::-1])
assert np.allclose(unproject(uv, z, calibration['K_left']), X)
T = make_transform(exp_so3([.1,-.2,.05]), [.3,.2,-.1])
restored = transform_points(inverse_transform(T), transform_points(T, X))
assert np.allclose(restored, X)

## Step 3

Report the numerical residual and the limits of the claim.

Before running: write your expected result and name one way this check could fail.

In [5]:
result = {'reference_points': len(X),
          'rigid_round_trip_max_m': float(np.max(np.linalg.norm(restored-X, axis=1))),
          'projection_round_trip': 'passed', 'depth_type': 'optical-axis Z',
          'independent_calibration_validated': False}

## Keep execution evidence separate from learning evidence

The following saves this reference calculation. Write your own explanation in `my_work/`; no learner score is assigned.

In [6]:
result['stage'] = 14
result['execution_role'] = 'reference_consolidation'
result['data_role'] = 'Real calibrated stereo-reference points plus a hand-checkable pinhole control.'
output = ROOT/'reports/consolidation_v4/14.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(result, indent=2, allow_nan=False) + '\n')
print('Saved reference result:', output.relative_to(ROOT))


Saved reference result: reports/consolidation_v4/14.json
